# nn-parameter-wrap — ex1: Parameter vs raw tensor — visibility test

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `nn-parameter-wrap`. Running the final beacon cell reports progress against the `PyTorch: nn.Parameter` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: nn.Parameter` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`nn-parameter-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "nn-parameter-wrap"
DD_SUBTOPIC = "PyTorch: nn.Parameter"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `nn.Parameter` — quick refresher

`nn.Parameter(tensor)` is a tensor subclass with one job: when assigned as an attribute of an `nn.Module`, it gets auto-registered in the module's `_parameters` dict so it shows up in `.parameters()`, `.state_dict()`, and gets moved by `.to(device)` / `.cuda()`.

**Parameter vs raw tensor.** `self.w = torch.randn(3)` — invisible to the optimizer. `self.w = nn.Parameter(torch.randn(3))` — included.

**Parameter vs buffer.** Both round-trip through `state_dict()`. Only Parameters are trainable (`.requires_grad=True` by default and included in `.parameters()`). Buffers are for non-learnable state — running stats, position encodings, attention masks. Register with `self.register_buffer('running_mean', torch.zeros(C))`.

**Gotcha — default dtype.** `nn.Parameter(torch.tensor([1, 2, 3]))` creates an `int64` parameter, which optimizers reject. Always pass a float tensor or call `.float()` first.

### Exercise 1 — Parameter vs raw tensor — visibility test

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Demonstrate that wrapping a tensor with nn.Parameter is what makes it visible to .parameters(), .state_dict(), and the optimizer — a raw tensor attribute is invisible.
> Keywords: nn.Parameter, parameters, state_dict, optimizer
> ```

**KCs targeted:** `parameter-wrap-tensor`, `parameter-auto-register`

Implement `ex1_build_module_pair()` returning a tuple `(invisible_mod, visible_mod)`.

Both Modules look almost identical — they each store a single tensor of shape `(4,)` as an attribute named `weight`. The ONLY difference:

1. `InvisibleMod.weight = t.randn(4)` — raw tensor. NOT wrapped.
2. `VisibleMod.weight = nn.Parameter(t.randn(4))` — wrapped.

Use `t.manual_seed(42)` immediately before EACH `randn` call so both Modules hold the same numeric values (the test ignores the values; this just keeps reproducibility tidy).

Both Modules must:
- Subclass `t.nn.Module`.
- Call `super().__init__()` in `__init__`.
- Implement `forward(self, x)` returning `x * self.weight`.

Return `(invisible_mod, visible_mod)` — both already instantiated.

The test asserts that `invisible_mod` has ZERO entries in `.parameters()` and an EMPTY `.state_dict()`, while `visible_mod` has exactly one parameter named `weight`. Same code structure, totally different visibility — that's the whole point of `nn.Parameter`.

In [ ]:
def ex1_build_module_pair():
    """Return (invisible_mod, visible_mod) — same shape, different visibility."""
    raise NotImplementedError()


def _test_ex1():
    invisible_mod, visible_mod = ex1_build_module_pair()

    # Both are nn.Modules.
    assert isinstance(invisible_mod, t.nn.Module)
    assert isinstance(visible_mod, t.nn.Module)

    # Invisible mod — raw tensor → registry empty.
    inv_params = list(invisible_mod.parameters())
    assert len(inv_params) == 0, (
        f'invisible_mod should have 0 params, got {len(inv_params)} '
        f'(did you wrap with nn.Parameter? you should NOT have, for invisible_mod)'
    )
    assert len(invisible_mod.state_dict()) == 0, 'invisible_mod state_dict should be empty'
    # Still callable.
    x = t.ones(4)
    y_inv = invisible_mod(x)
    assert y_inv.shape == (4,)

    # Visible mod — nn.Parameter → registered.
    vis_params = list(visible_mod.parameters())
    assert len(vis_params) == 1, (
        f'visible_mod should have exactly 1 param, got {len(vis_params)} '
        f'(did you wrap weight with nn.Parameter?)'
    )
    assert isinstance(vis_params[0], t.nn.Parameter)
    assert vis_params[0].shape == (4,)
    named = dict(visible_mod.named_parameters())
    assert 'weight' in named, f'expected param named "weight", got {list(named.keys())}'
    assert 'weight' in visible_mod.state_dict(), 'weight should round-trip via state_dict'
    y_vis = visible_mod(x)
    assert y_vis.shape == (4,)

    # requires_grad True by default for nn.Parameter, False (or absent) for raw tensor.
    assert vis_params[0].requires_grad, 'nn.Parameter defaults to requires_grad=True'

    # The smoking gun: SGD over invisible_mod has nothing to optimize.
    try:
        t.optim.SGD(invisible_mod.parameters(), lr=0.1)
        raise RuntimeError('SGD should reject an empty parameter list')
    except ValueError as e:
        assert 'empty parameter list' in str(e) or 'got an empty' in str(e), (
            f'expected ValueError about empty parameter list, got: {e}'
        )
        print(f'  observed expected SGD rejection: {e}')
    # But SGD over visible_mod is fine.
    opt = t.optim.SGD(visible_mod.parameters(), lr=0.1)
    assert len(opt.param_groups[0]['params']) == 1
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_build_module_pair():
    class InvisibleMod(t.nn.Module):
        def __init__(self):
            super().__init__()
            t.manual_seed(42)
            self.weight = t.randn(4)  # raw tensor — NOT a Parameter
        def forward(self, x):
            return x * self.weight
    class VisibleMod(t.nn.Module):
        def __init__(self):
            super().__init__()
            t.manual_seed(42)
            self.weight = t.nn.Parameter(t.randn(4))  # wrapped → registered
        def forward(self, x):
            return x * self.weight
    return InvisibleMod(), VisibleMod()
```

**The mechanism.** `nn.Module.__setattr__` inspects the value of every attribute assignment. If it's a `Parameter`, it stores it in `self._parameters` (the dict `.parameters()` iterates). If it's a raw `Tensor`, it gets stored on `self.__dict__` like any Python attribute — invisible to `.parameters()`.

**Why this is a footgun.** Forward still works fine with a raw tensor — you only notice the bug when training silently doesn't improve, because the optimizer never sees the weight. Always wrap learnable tensors in `nn.Parameter`.

**The non-learnable case.** If you have a tensor that should round-trip via `state_dict` but NOT be trained (e.g. attention mask, running mean), use `self.register_buffer('name', tensor)` instead. That's the third option — and exactly what `ex2` explores.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()